In [1]:
import os
import sys
import gzip
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from torch.optim import Adam

In [2]:
src_path = os.path.abspath('../src') 
sys.path.append(src_path)

In [3]:
from atac_dataset import create_dataloader
from model import DAE

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
dataloader = create_dataloader(dense_dir = "/data1/peerd/paddockl/projects/DL_project/ATAC_denoising/data/ENCODE_ATAC_chr19_Counts",
                               sparse_dir = "/data1/peerd/paddockl/projects/DL_project/ATAC_denoising/data/atac_chr19_subsampled",
                               batch_size = 7)

In [6]:
x, y = next(iter(dataloader))

In [7]:
y.shape

torch.Size([7, 109792])

In [8]:
model = DAE().to(device)

In [9]:
model

DAE(
  (encoder): Sequential(
    (0): Linear(in_features=109792, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=64, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1024, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1024, out_features=109792, bias=True)
    (5): ReLU()
  )
)

In [10]:
# model = DAE(
#     input_dim=input_dim,
#     hidden_dims=[512, 128],
#     latent_dim=32).to(device)

In [11]:
loss_fn = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=1e-3)

epochs = 100

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for x, y in dataloader:

        x = x.to(device).float()
        y = y.to(device).float()

        recon = model(x)

        loss = loss_fn(recon, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

Epoch 1 | Loss: 4659.6710
Epoch 2 | Loss: 2651.4838
Epoch 3 | Loss: 2947.9880
Epoch 4 | Loss: 1984.3368
Epoch 5 | Loss: 2019.0602
Epoch 6 | Loss: 1772.5519
Epoch 7 | Loss: 1771.9630
Epoch 8 | Loss: 1986.4821
Epoch 9 | Loss: 1489.7973
Epoch 10 | Loss: 1849.4120
Epoch 11 | Loss: 1876.0994
Epoch 12 | Loss: 1383.9618
Epoch 13 | Loss: 1463.0938
Epoch 14 | Loss: 1688.1063
Epoch 15 | Loss: 1455.0039
Epoch 16 | Loss: 2149.9863
Epoch 17 | Loss: 2433.5502
Epoch 18 | Loss: 1747.0569
Epoch 19 | Loss: 1650.8129
Epoch 20 | Loss: 1729.3607
Epoch 21 | Loss: 1878.1043
Epoch 22 | Loss: 2167.5895
Epoch 23 | Loss: 2455.4302
Epoch 24 | Loss: 1915.9946
Epoch 25 | Loss: 1568.3226
Epoch 26 | Loss: 1391.8458
Epoch 27 | Loss: 1224.6964
Epoch 28 | Loss: 1281.9642
Epoch 29 | Loss: 2286.2828
Epoch 30 | Loss: 1899.2594
Epoch 31 | Loss: 1955.7522
Epoch 32 | Loss: 1297.8433
Epoch 33 | Loss: 2350.4123
Epoch 34 | Loss: 1757.8196
Epoch 35 | Loss: 2025.4424
Epoch 36 | Loss: 2104.8713
Epoch 37 | Loss: 1550.4821
Epoch 38 |